# Simulation ALS Data: Two-Epoch TAM3C2 + Multi-Scale Analysis

Time-Adaptive M3C2 using `py4dgeo.tam3c2` on one reference epoch and one explicitly selected target epoch.

**Dataset:** 
- **Location:** `C:\rsa\research_proj\blender_project\simulation_test2\output\simulation_test2_als`
- **Format:** XYZ files

**Workflow:**
1. Load all available epochs
2. Select one reference epoch and one target epoch
3. Sample corepoints from the reference epoch
4. Build a `TAM3C2` algorithm object with only the reference/target pair in `epochs_timeseries`
5. Run per-target TAM3C2 for the selected target epoch
6. Compare multi-scale spherical normal estimation and cylindrical distance-estimation behavior

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from datetime import datetime, timedelta

import numpy as np
import matplotlib.pyplot as plt

import py4dgeo
from py4dgeo import (
    TAM3C2,
    Weighting,
    extract_reference_and_others,
    sample_corepoints,
)
from py4dgeo.segmentation import RegionGrowingAlgorithm, RegionGrowingSeed, temporal_averaging
from py4dgeo.data_loader import read_pc_epochs_and_assign_timestamps

## 1. Configuration

In [ ]:
data_path = r'C:\rsa\research_proj\blender_project\simulation_test2\output\simulation_test2_als_downsampled1'

# Two-epoch TAM3C2 setup: one reference epoch and one explicit target epoch.
reference_timestamp = datetime(2020, 1, 6, 0, 0, 0)
target_timestamp = datetime(2020, 1, 22, 0, 0, 0)

# Spatial block selection. Use "all" for the full scene, or 1-4 for one 40x40 m block.
# Full scene x/y range: [-20, 60]. Block edges are x=-20/20/60 and y=-20/20/60.
# Numbering: 1=lower-left, 2=lower-right, 3=upper-left, 4=upper-right.
selected_block = 'all'
scene_x_range = (-20.0, 60.0)
scene_y_range = (-20.0, 60.0)
scene_subset_label = f"block{selected_block}" if selected_block != "all" else "all"

# Input archive and generated analysis archives
notebook_directory = os.getcwd()
dataset_name = os.path.basename(os.path.normpath(data_path))
output_directory = os.path.join(notebook_directory, dataset_name)
os.makedirs(output_directory, exist_ok=True)
reference_file_path = os.path.join(
    notebook_directory,
    "simulation2_reference.zip",
)
single_target_output_path = os.path.join(
    output_directory,
    f"simulation2_als_{scene_subset_label}_single_target_multiscale_tam3c2.zip",
)
timeseries_best_scale_output_template = os.path.join(
    output_directory,
    f"simulation2_als_{scene_subset_label}_full_timeseries_best_scale_idx{{best_combo_idx}}_weighted_tam3c2.zip",
)
timeseries_standard_m3c2_output_template = os.path.join(
    output_directory,
    f"simulation2_als_{scene_subset_label}_full_timeseries_best_scale_idx{{best_combo_idx}}_standard_m3c2.zip",
)
best_scale_output_template = os.path.join(
    output_directory,
    f"simulation2_als_{scene_subset_label}_single_target_best_scale_idx{{best_combo_idx}}_weighted_tam3c2.zip",
)
no_weight_output_template = os.path.join(
    output_directory,
    f"simulation2_als_{scene_subset_label}_single_target_best_scale_idx{{best_combo_idx}}_unweighted_tam3c2.zip",
)
temporal_only_output_template = os.path.join(
    output_directory,
    f"simulation2_als_{scene_subset_label}_single_target_best_scale_idx{{best_combo_idx}}_temporal_only_tam3c2.zip",
)
direct_m3c2_output_template = os.path.join(
    output_directory,
    f"simulation2_als_{scene_subset_label}_single_target_best_scale_idx{{best_combo_idx}}_direct_m3c2.zip",
)
fixed_archive_output_path = os.path.join(
    output_directory,
    f"simulation2_als_{scene_subset_label}_single_target_fixed_normal_m3c2.zip",
)

# TAM3C2 parameters - multi-scale grid for reference/target-only aggregation
normal_radii = [0.1, 0.2, 0.3, 0.5]
max_window_ratio = [0.2, 0.3, 0.5]
required_points = 10
cyl_radius = 1.0
max_distance = 10.0
registration_error = 0.01
sigma_ratio = 1.0
space_time_ratio = 1.0
weighting = Weighting.GAUSSIAN
include_center_epoch = True
keep_neighborhoods = True

# 4D-OBC parameters
obc_neighborhood_radius = 1.0
obc_min_segments = 10
obc_minperiod = 3
obc_height_threshold = 0.05
obc_thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]
obc_seed_subsampling = 1

### corepoints from the reference file

In [ ]:
ref_analysis = py4dgeo.SpatiotemporalAnalysis(reference_file_path, force=False)
all_corepoints = ref_analysis.corepoints.cloud

x_min, x_max = scene_x_range
y_min, y_max = scene_y_range
x_mid = 0.5 * (x_min + x_max)
y_mid = 0.5 * (y_min + y_max)

block_bounds = {
    1: (x_min, x_mid, y_min, y_mid),
    2: (x_mid, x_max, y_min, y_mid),
    3: (x_min, x_mid, y_mid, y_max),
    4: (x_mid, x_max, y_mid, y_max),
}

if selected_block == "all":
    corepoints = all_corepoints
    print(f"Selected full scene: x=[{x_min}, {x_max}], y=[{y_min}, {y_max}]")
    print(f"Corepoints used: {len(corepoints):,}/{len(all_corepoints):,}")
else:
    if selected_block not in block_bounds:
        raise ValueError(f"selected_block must be 'all' or one of {sorted(block_bounds)}, got {selected_block}")

    block_x_min, block_x_max, block_y_min, block_y_max = block_bounds[selected_block]
    x_in_block = (all_corepoints[:, 0] >= block_x_min) & (
        all_corepoints[:, 0] < block_x_max if selected_block in (1, 3) else all_corepoints[:, 0] <= block_x_max
    )
    y_in_block = (all_corepoints[:, 1] >= block_y_min) & (
        all_corepoints[:, 1] < block_y_max if selected_block in (1, 2) else all_corepoints[:, 1] <= block_y_max
    )
    block_mask = x_in_block & y_in_block

    corepoints = all_corepoints[block_mask]
    if len(corepoints) == 0:
        raise ValueError(
            f"No corepoints found in block {selected_block}: "
            f"x=[{block_x_min}, {block_x_max}], y=[{block_y_min}, {block_y_max}]"
        )

    print(f"Selected block {selected_block}: x=[{block_x_min}, {block_x_max}], y=[{block_y_min}, {block_y_max}]")
    print(f"Corepoints in selected block: {len(corepoints):,}/{len(all_corepoints):,}")

## 2. Load epochs and select the reference/target pair

In [ ]:
epochs = read_pc_epochs_and_assign_timestamps(folder=data_path, start_time=datetime(2020, 1, 1), time_increment=timedelta(days=1))

In [ ]:
# Keep only the two epochs used by TAM3C2 aggregation.
epochs_all = sorted(epochs, key=lambda e: e.timestamp)
reference_epoch = next((e for e in epochs_all if e.timestamp == reference_timestamp), None)
target_epoch = next((e for e in epochs_all if e.timestamp == target_timestamp), None)

if reference_epoch is None:
    raise ValueError(f"Reference {reference_timestamp} not in data")

if target_epoch is None:
    raise ValueError(f"Target {target_timestamp} not in data")

if target_epoch.timestamp == reference_epoch.timestamp:
    raise ValueError("target_timestamp must differ from reference_timestamp")

## 3. Build TAM3C2 and run the spatiotemporal analysis

In [ ]:
tam = TAM3C2(
    epochs_timeseries=epochs_all,
    max_window_ratio=max_window_ratio,
    normal_radii=normal_radii,
    required_points=required_points,
    weighting=weighting,
    sigma_ratio=sigma_ratio,
    space_time_ratio=space_time_ratio,
    include_center_epoch=include_center_epoch,
    keep_neighborhoods=keep_neighborhoods,
    corepoints=corepoints,
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

single_target_multiscale_analysis = py4dgeo.SpatiotemporalAnalysis(
    single_target_output_path,
    force=True,
)
single_target_multiscale_analysis.reference_epoch = reference_epoch
single_target_multiscale_analysis.corepoints = corepoints
single_target_multiscale_analysis.m3c2 = tam

single_target_multiscale_analysis.add_epochs(target_epoch)
print(f"target epoch: {target_epoch.timestamp}")
print(f"epochs_timeseries used by TAM3C2: {[e.timestamp for e in tam.epochs_timeseries]}")
print(f"include_center_epoch: {tam.include_center_epoch}")
print(f"distances shape: {single_target_multiscale_analysis.distances.shape}")
print(f"uncertainties shape: {single_target_multiscale_analysis.uncertainties.shape}")

## 4. statistic aggregation diagnostics for the selected target

In [ ]:
diag = tam.diagnostics()

## visual analysis of aggregation

In [ ]:
# Visual diagnostics for temporal aggregation around each corepoint.
# Requires `tam`, `diag`, `reference_epoch`, `target_epoch`, and `corepoints` from previous cells.

def _diag_col(name, col=0):
    value = diag[name]
    arr = np.asarray(value)
    return arr[:, col] if arr.ndim == 2 else arr


def _plot_cp_map(ax, values, title, cmap='viridis', s=3, vmin=None, vmax=None, discrete=False):
    sc = ax.scatter(corepoints[:, 0], corepoints[:, 1], c=values, s=s, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel('X [m]')
    ax.set_ylabel('Y [m]')
    ax.axis('equal')
    cbar = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
    if discrete:
        vals = np.unique(np.asarray(values)[np.isfinite(values)])
        if len(vals) <= 12:
            cbar.set_ticks(vals)
    return sc


def _count_cylinder_points_in_epoch(cp, normal, epoch_idx, cyl_radius, max_distance):
    bounding_r = np.sqrt(cyl_radius * cyl_radius + max_distance * max_distance)
    idxs = tam._kdtrees[epoch_idx].query_ball_point(cp, bounding_r)
    if not idxs:
        return 0
    pts = tam.epochs_timeseries[epoch_idx].cloud[idxs]
    vec = pts - cp
    along = vec @ normal
    perp_sq = np.einsum('ij,ij->i', vec, vec) - along * along
    mask = (perp_sq <= cyl_radius * cyl_radius) & (np.abs(along) <= max_distance)
    return int(np.count_nonzero(mask))


tam._build_index()
target_col = 0
scale_idx = np.asarray(diag['scale_idx']).astype(int)
combos = tam._scale_combinations

ref_idx_in_ts = tam._find_epoch_index(reference_epoch)
tgt_idx_in_ts = tam._find_epoch_index(target_epoch)
ref_time = reference_epoch.timestamp.timestamp()
time_range = float(tam._epoch_times.max() - tam._epoch_times.min()) or 1.0
ref_exclude_idx = None if tam.include_center_epoch else ref_idx_in_ts

n_points_ref_after = _diag_col('n_points_ref', target_col)
n_points_tgt_after = _diag_col('n_points_tgt', target_col)

# Exact number of contributing epochs if keep_neighborhoods=True; fallback to diagnostics otherwise.
nbhd = tam._neighborhoods[target_col] if tam._neighborhoods is not None else None
if nbhd is not None:
    n_epochs_ref = np.array([
        len(np.unique(record['ref_eidx'])) if record is not None else 0
        for record in nbhd
    ], dtype=float)
    n_epochs_tgt = np.array([
        len(np.unique(record['tgt_eidx'])) if record is not None else 0
        for record in nbhd
    ], dtype=float)
else:
    n_epochs_ref = _diag_col('n_before_ref', target_col) + _diag_col('n_after_ref', target_col)
    n_epochs_tgt = _diag_col('n_before_tgt', target_col) + _diag_col('n_after_tgt', target_col)
    print('keep_neighborhoods=False: epoch-count maps use n_before + n_after diagnostics.')

# Center-epoch-only cylinder counts before temporal aggregation.
n_points_ref_before = np.zeros(len(corepoints), dtype=int)
n_points_tgt_before = np.zeros(len(corepoints), dtype=int)
planarity_selected = np.full(len(corepoints), np.nan)

for i, cp in enumerate(corepoints):
    normal = tam._ref_normals[i]
    sr, wr = combos[scale_idx[i]]

    n_points_ref_before[i] = _count_cylinder_points_in_epoch(
        cp, normal, ref_idx_in_ts, tam.cyl_radius, tam.max_distance
    )
    n_points_tgt_before[i] = _count_cylinder_points_in_epoch(
        cp, normal, tgt_idx_in_ts, tam.cyl_radius, tam.max_distance
    )

    pts, *_ = tam._aggregate_sphere(cp, ref_time, ref_exclude_idx, sr, time_range * wr)
    if pts is not None and len(pts) >= 3:
        planarity_selected[i], _ = tam._planarity_and_normal(pts)

# Required-points status: 0=neither side, 1=ref only, 2=target only, 3=both sides.
required_before = (
    (n_points_ref_before >= required_points).astype(int)
    + 2 * (n_points_tgt_before >= required_points).astype(int)
)
required_after = (
    (n_points_ref_after >= required_points).astype(int)
    + 2 * (n_points_tgt_after >= required_points).astype(int)
)

print('Scale index -> (normal_radius, max_window_ratio)')
for idx, combo in enumerate(combos):
    print(f'  {idx}: {combo}')
print('Required status code: 0=neither side, 1=ref only, 2=target only, 3=both sides')

# 1. Aggregation epoch-count maps.
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
vmax_epochs = max(np.nanmax(n_epochs_ref), np.nanmax(n_epochs_tgt))
_plot_cp_map(axs[0], n_epochs_ref, 'Reference aggregation: contributing epochs', 'viridis', vmax=vmax_epochs, discrete=True)
_plot_cp_map(axs[1], n_epochs_tgt, 'Target aggregation: contributing epochs', 'viridis', vmax=vmax_epochs, discrete=True)
plt.tight_layout()
plt.show()

# 2. Point-count maps before vs after temporal aggregation.
fig, axs = plt.subplots(2, 2, figsize=(13, 10))
vmax_points = np.nanpercentile(
    np.r_[n_points_ref_before, n_points_tgt_before, n_points_ref_after, n_points_tgt_after],
    98,
)
_plot_cp_map(axs[0, 0], n_points_ref_before, 'Reference center epoch: cylinder points', 'magma', vmax=vmax_points)
_plot_cp_map(axs[0, 1], n_points_ref_after, 'Reference after temporal aggregation: points used', 'magma', vmax=vmax_points)
_plot_cp_map(axs[1, 0], n_points_tgt_before, 'Target center epoch: cylinder points', 'magma', vmax=vmax_points)
_plot_cp_map(axs[1, 1], n_points_tgt_after, 'Target after temporal aggregation: points used', 'magma', vmax=vmax_points)
plt.tight_layout()
plt.show()

# 3. Required-points status before and after aggregation.
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
_plot_cp_map(axs[0], required_before, f'Before aggregation: required_points={required_points}', 'tab10', vmin=0, vmax=3, discrete=True)
_plot_cp_map(axs[1], required_after, f'After aggregation: required_points={required_points}', 'tab10', vmin=0, vmax=3, discrete=True)
plt.tight_layout()
plt.show()

# 4. Selected spatiotemporal scale and planarity.
# Bivariate color mixture: blue = selected normal radius, red = selected max_window_ratio.
selected_radius = np.array([combos[idx][0] for idx in scale_idx], dtype=float)
selected_window_ratio = np.array([combos[idx][1] for idx in scale_idx], dtype=float)


def _normalize_to_unit(values):
    values = np.asarray(values, dtype=float)
    scaled = np.zeros_like(values, dtype=float)
    finite = np.isfinite(values)
    if not np.any(finite):
        return scaled, np.nan, np.nan
    vmin = float(np.nanmin(values[finite]))
    vmax = float(np.nanmax(values[finite]))
    if vmax > vmin:
        scaled[finite] = (values[finite] - vmin) / (vmax - vmin)
    else:
        scaled[finite] = 0.5
    scaled[~finite] = np.nan
    return scaled, vmin, vmax


def _mix_scale_colors(radius_level, ratio_level):
    radius_level = np.asarray(radius_level, dtype=float)
    ratio_level = np.asarray(ratio_level, dtype=float)
    low = np.array([0.93, 0.93, 0.93])
    blue = np.array([0.08, 0.30, 0.95])
    red = np.array([0.95, 0.08, 0.28])
    purple = np.array([0.28, 0.00, 0.45])

    radius = np.nan_to_num(radius_level, nan=0.0)[..., None]
    ratio = np.nan_to_num(ratio_level, nan=0.0)[..., None]
    return (
        (1.0 - radius) * (1.0 - ratio) * low
        + radius * (1.0 - ratio) * blue
        + (1.0 - radius) * ratio * red
        + radius * ratio * purple
    )


radius_level, radius_min, radius_max = _normalize_to_unit(selected_radius)
ratio_level, ratio_min, ratio_max = _normalize_to_unit(selected_window_ratio)
mixed_colors = _mix_scale_colors(radius_level, ratio_level)

fig = plt.figure(figsize=(17, 5.8), constrained_layout=True)
gs = fig.add_gridspec(1, 3, width_ratios=[1.15, 0.55, 1.15])
ax_scale = fig.add_subplot(gs[0, 0])
ax_key = fig.add_subplot(gs[0, 1])
ax_planarity = fig.add_subplot(gs[0, 2])

ax_scale.scatter(
    corepoints[:, 0],
    corepoints[:, 1],
    c=mixed_colors,
    s=6,
    linewidths=0,
    alpha=0.9,
)
ax_scale.set_title('Selected spatiotemporal scale')
ax_scale.set_xlabel('X [m]')
ax_scale.set_ylabel('Y [m]')
ax_scale.set_aspect('equal', adjustable='box')

key_steps = 120
radius_grid = np.linspace(0.0, 1.0, key_steps)
ratio_grid = np.linspace(0.0, 1.0, key_steps)
R, T = np.meshgrid(radius_grid, ratio_grid)
key_colors = _mix_scale_colors(R, T)
ax_key.imshow(
    key_colors,
    origin='lower',
    extent=[radius_min, radius_max, ratio_min, ratio_max],
    aspect='auto',
)
ax_key.set_title('Color mixture')
ax_key.set_xlabel('normal radius [m]')
ax_key.set_ylabel('max_window_ratio')
ax_key.set_xticks(sorted(set(selected_radius)))
ax_key.set_yticks(sorted(set(selected_window_ratio)))
ax_key.tick_params(labelsize=8)

_plot_cp_map(ax_planarity, planarity_selected, 'Planarity at selected scale', 'viridis')
plt.show()

## 5. Multi-scale analysis 

In [ ]:
# Full-corepoint multi-scale planarity map on the (normal_radii x max_window_ratio) grid.
#
# Important: the mean-planarity panel below does NOT run full TAM3C2 once per
# combo. It recomputes the spherical normal-estimation neighborhood for every
# (scale combo, corepoint) pair and reports the PCA planarity. This is the same
# criterion used by TAM3C2's scale-selection contest.
#
# The selection-frequency panel comes from the already-run multiscale TAM3C2
# cache: each corepoint votes for the combo selected by TAM3C2.
#
# The selected best single-scale combo is then run once as a separate TAM3C2
# analysis and saved for later comparison.

radii  = list(tam.normal_radii)     if hasattr(tam.normal_radii,     '__iter__') else [tam.normal_radii]
ratios = list(tam.max_window_ratio) if hasattr(tam.max_window_ratio, '__iter__') else [tam.max_window_ratio]
combos = tam._scale_combinations    # ordered: for sr in radii: for wr in ratios

ref_idx_in_ts = tam._find_epoch_index(reference_epoch)
ref_exclude_idx = None if tam.include_center_epoch else ref_idx_in_ts
ref_time = reference_epoch.timestamp.timestamp()
time_range = float(tam._epoch_times.max() - tam._epoch_times.min()) or 1.0

# --- 1. planarity for every (combo, corepoint), no subsampling --------------
n_corepoints = len(corepoints)
planarity = np.full((len(combos), n_corepoints), np.nan)

for k, (sr, wr) in enumerate(combos):
    mw = time_range * wr
    print(f"Computing planarity for combo {k}/{len(combos)-1}: normal_radius={sr:g}, max_window_ratio={wr:g}")
    for i, cp in enumerate(corepoints):
        pts, *_ = tam._aggregate_sphere(cp, ref_time, ref_exclude_idx, sr, mw)
        if pts is None or len(pts) < 3:
            continue
        pl, _ = tam._planarity_and_normal(pts)
        planarity[k, i] = pl

mean_pl_flat = np.nanmean(planarity, axis=1)
mean_pl = mean_pl_flat.reshape(len(radii), len(ratios))

# --- 2. selection frequency from cached multiscale diagnostics --------------
diag = tam.diagnostics()
scale_idx_all = np.asarray(diag['scale_idx']).astype(int)
counts = np.bincount(scale_idx_all, minlength=len(combos))
freq_flat = counts / counts.sum()
freq = freq_flat.reshape(len(radii), len(ratios))
combo_idx_grid = np.arange(len(combos)).reshape(len(radii), len(ratios))

# --- 3. choose one best combo and save a single-scale TAM3C2 result ----------
best_mean_planarity_idx = int(np.nanargmax(mean_pl_flat))
most_selected_idx = int(np.argmax(counts))

# Default strategy: choose by mean planarity. If several combinations are tied
# within this tolerance, prefer the smaller spatial radius and then the smaller
# temporal window ratio.
best_combo_strategy = "mean_planarity" #！！！
planarity_tie_tolerance = 1e-3

if best_combo_strategy == "mean_planarity":
    best_mean_planarity = float(np.nanmax(mean_pl_flat))
    tied_mean_planarity_idx = np.flatnonzero(
        np.isfinite(mean_pl_flat)
        & (mean_pl_flat >= best_mean_planarity - planarity_tie_tolerance)
    )
    if len(tied_mean_planarity_idx) == 0:
        raise ValueError("No finite mean planarity value found.")
    best_combo_idx = int(
        min(
            tied_mean_planarity_idx,
            key=lambda idx: (combos[int(idx)][0], combos[int(idx)][1]),
        )
    )
elif best_combo_strategy == "most_selected":
    best_combo_idx = most_selected_idx
else:
    raise ValueError(f"Unknown best_combo_strategy: {best_combo_strategy!r}")

best_sr, best_wr = combos[best_combo_idx]
timeseries_output_path = timeseries_best_scale_output_template.format(
    best_combo_idx=best_combo_idx
)
timeseries_standard_m3c2_output_path = (
    timeseries_standard_m3c2_output_template.format(
        best_combo_idx=best_combo_idx
    )
)
best_scale_output_path = best_scale_output_template.format(
    best_combo_idx=best_combo_idx
)

print("\nScale index -> (normal_radius, max_window_ratio)")
for idx, combo in enumerate(combos):
    print(f"  {idx}: {combo}")
print(f"\nBest by mean planarity : idx={best_mean_planarity_idx}, combo={combos[best_mean_planarity_idx]}, mean_planarity={mean_pl_flat[best_mean_planarity_idx]:.4f}")
print(f"Best by selection freq : idx={most_selected_idx}, combo={combos[most_selected_idx]}, frequency={freq_flat[most_selected_idx]:.1%}")
if best_combo_strategy == "mean_planarity":
    print(
        "Mean-planarity tie candidates "
        f"(within {planarity_tie_tolerance:g}): "
        f"{[(int(idx), combos[int(idx)], float(mean_pl_flat[int(idx)])) for idx in tied_mean_planarity_idx]}"
    )
print(f"Using best_combo_strategy={best_combo_strategy!r}: idx={best_combo_idx}, combo=({best_sr}, {best_wr})")

best_scale_tam = TAM3C2(
    epochs_timeseries=epochs_all,
    max_window_ratio=float(best_wr),
    normal_radii=float(best_sr),
    required_points=required_points,
    weighting=weighting,
    sigma_ratio=sigma_ratio,
    space_time_ratio=space_time_ratio,
    include_center_epoch=include_center_epoch,
    keep_neighborhoods=False,
    corepoints=corepoints,
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

best_scale_analysis = py4dgeo.SpatiotemporalAnalysis(best_scale_output_path, force=True)
best_scale_analysis.reference_epoch = reference_epoch
best_scale_analysis.corepoints = corepoints
best_scale_analysis.m3c2 = best_scale_tam
best_scale_analysis.add_epochs(target_epoch)

multi_dist = single_target_multiscale_analysis.distances[:, 0]
best_dist = best_scale_analysis.distances[:, 0]
compare_valid = np.isfinite(multi_dist) & np.isfinite(best_dist)
print(f"Saved best single-scale TAM3C2 analysis: {best_scale_output_path}")
print(f"Best single-scale distances shape: {best_scale_analysis.distances.shape}")
print(f"Compared to per-corepoint multiscale TAM3C2 on {compare_valid.sum()} valid corepoints:")
print(f"  mean(best - multiscale) = {np.nanmean(best_dist[compare_valid] - multi_dist[compare_valid]):.4f} m")
print(f"  MAE(best vs multiscale) = {np.nanmean(np.abs(best_dist[compare_valid] - multi_dist[compare_valid])):.4f} m")

# --- 4. two-panel heatmap with combo index labels ---------------------------
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for ax, M, title, cmap, fmt in [
    (axs[0], mean_pl, 'Mean planarity over all corepoints', 'viridis', '.3f'),
    (axs[1], freq,    'Selection frequency of winning scale', 'magma', '.1%'),
]:
    im = ax.imshow(M, origin='lower', aspect='auto', cmap=cmap)
    ax.set_xticks(range(len(ratios)))
    ax.set_xticklabels([f'{r:g}' for r in ratios])
    ax.set_yticks(range(len(radii)))
    ax.set_yticklabels([f'{r:g}' for r in radii])
    ax.set_xlabel('max_window_ratio')
    ax.set_ylabel('normal_radii [m]')
    ax.set_title(title)
    for row in range(M.shape[0]):
        for col in range(M.shape[1]):
            val = M[row, col]
            if not np.isnan(val):
                ax.text(
                    col,
                    row,
                    format(val, fmt),
                    ha='center',
                    va='center',
                    color='white',
                    fontsize=9,
                )
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

for ax in axs:
    best_row, best_col = np.argwhere(combo_idx_grid == best_combo_idx)[0]
    ax.scatter(best_col, best_row, s=220, facecolors='none', edgecolors='cyan', linewidths=2.5)

plt.tight_layout()
plt.show()

## 7. Compare against reference distances from `simulation2_reference.zip`

Load the precomputed analysis archive, select the column matching the current target epoch, align corepoints, and compare the current TAM3C2 distance estimates against the reference distances.

In [ ]:
indices = np.nonzero(ref_analysis.distances)[0]
np.unique(indices)

In [ ]:
# Recompute and save the no-weight best single-scale TAM3C2 analysis.
# This uses the best scale selected in the previous multiscale-analysis cell,
# but turns temporal weighting off for both reference and target aggregation.

no_weight_output_path = no_weight_output_template.format(
    best_combo_idx=best_combo_idx
)

no_weight_tam = TAM3C2(
    epochs_timeseries=epochs_all,
    max_window_ratio=float(best_wr),
    normal_radii=float(best_sr),
    required_points=required_points,
    weighting=Weighting.NONE,
    sigma_ratio=sigma_ratio,
    space_time_ratio=space_time_ratio,
    include_center_epoch=include_center_epoch,
    keep_neighborhoods=False,
    corepoints=corepoints,
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

no_weight_analysis = py4dgeo.SpatiotemporalAnalysis(no_weight_output_path, force=True)
no_weight_analysis.reference_epoch = reference_epoch
no_weight_analysis.corepoints = corepoints
no_weight_analysis.m3c2 = no_weight_tam
no_weight_analysis.add_epochs(target_epoch)


print(f"Saved unweighted best-scale TAM3C2 analysis: {no_weight_output_path}")
print(f"Unweighted distances shape: {no_weight_analysis.distances.shape}")
print(f"Unweighted uncertainties shape: {no_weight_analysis.uncertainties.shape}")

In [ ]:
# Recompute and save the temporal-only weighted best single-scale TAM3C2 analysis.
# This keeps the Gaussian temporal weighting but disables the spatial part of
# the weighting kernel, so auxiliary epochs are down-weighted only by time.

temporal_only_output_path = temporal_only_output_template.format(
    best_combo_idx=best_combo_idx
)

temporal_only_tam = TAM3C2(
    epochs_timeseries=epochs_all,
    max_window_ratio=float(best_wr),
    normal_radii=float(best_sr),
    required_points=required_points,
    weighting=weighting,
    sigma_ratio=sigma_ratio,
    space_time_ratio=space_time_ratio,
    spatial_weighting=False,
    include_center_epoch=include_center_epoch,
    keep_neighborhoods=False,
    corepoints=corepoints,
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

temporal_only_analysis = py4dgeo.SpatiotemporalAnalysis(temporal_only_output_path, force=True)
temporal_only_analysis.reference_epoch = reference_epoch
temporal_only_analysis.corepoints = corepoints
temporal_only_analysis.m3c2 = temporal_only_tam
temporal_only_analysis.add_epochs(target_epoch)

print(f"Saved temporal-only weighted best-scale TAM3C2 analysis: {temporal_only_output_path}")
print(f"Temporal-only weighted distances shape: {temporal_only_analysis.distances.shape}")
print(f"Temporal-only weighted uncertainties shape: {temporal_only_analysis.uncertainties.shape}")

## fixed normal

In [ ]:
# Shared Direct M3C2 setup.
direct_m3c2_normal_radius = float(best_sr)
direct_m3c2_kwargs = {
    "epochs": (reference_epoch, target_epoch),
    "corepoints": corepoints,
    "normal_radii": [direct_m3c2_normal_radius],
    "cyl_radius": cyl_radius,
    "max_distance": max_distance,
    "registration_error": registration_error,
}

fixed_m3c2_directions = np.asarray(
    py4dgeo.M3C2(**direct_m3c2_kwargs).directions(),
    dtype=float,
).copy()

print(f"Direct M3C2 normal radius: {direct_m3c2_normal_radius:g} m")
print(f"Fixed normal directions: {fixed_m3c2_directions.shape}")

In [ ]:
# Standard Direct M3C2 baseline, saved through SpatiotemporalAnalysis.
direct_m3c2_output_path = direct_m3c2_output_template.format(
    best_combo_idx=best_combo_idx
)

direct_m3c2_analysis = py4dgeo.SpatiotemporalAnalysis(
    direct_m3c2_output_path,
    force=True,
)
direct_m3c2_analysis.reference_epoch = reference_epoch
direct_m3c2_analysis.corepoints = corepoints
direct_m3c2_analysis.m3c2 = py4dgeo.M3C2(**direct_m3c2_kwargs)
direct_m3c2_analysis.add_epochs(target_epoch)

print(f"Saved direct M3C2 analysis: {direct_m3c2_output_path}")
print(f"Direct M3C2 distances shape: {direct_m3c2_analysis.distances.shape}")
print(f"Direct M3C2 uncertainties shape: {direct_m3c2_analysis.uncertainties.shape}")

In [ ]:
# Same standard Direct M3C2 baseline, returned directly from run().
m3c2_distances, m3c2_uncertainties = py4dgeo.M3C2(
    **direct_m3c2_kwargs,
).run()

print(f"Direct M3C2 run() distances shape: {m3c2_distances.shape}")
print(f"Direct M3C2 run() uncertainties shape: {m3c2_uncertainties.shape}")

In [ ]:
# Fixed-normal control: reuse one normal array for both execution paths.
class FixedDirectionsM3C2(py4dgeo.M3C2):
    def __init__(self, *args, fixed_directions, **kwargs):
        super().__init__(*args, **kwargs)
        self.fixed_directions = np.asarray(fixed_directions, dtype=float).copy()

    def directions(self):
        return self.fixed_directions.copy()


def fixed_normal_m3c2():
    return FixedDirectionsM3C2(
        **direct_m3c2_kwargs,
        fixed_directions=fixed_m3c2_directions,
    )


fixed_run_distances, fixed_run_uncertainties = fixed_normal_m3c2().run()
fixed_run_distances = np.asarray(fixed_run_distances, dtype=float).copy()
fixed_run_uncertainties = np.asarray(fixed_run_uncertainties).copy()

fixed_archive_analysis = py4dgeo.SpatiotemporalAnalysis(
    fixed_archive_output_path,
    force=True,
)
fixed_archive_analysis.reference_epoch = reference_epoch
fixed_archive_analysis.corepoints = corepoints
fixed_archive_analysis.m3c2 = fixed_normal_m3c2()
fixed_archive_analysis.add_epochs(target_epoch)

fixed_archive_distances = fixed_archive_analysis.distances[:, 0].astype(float).copy()
fixed_archive_uncertainties = fixed_archive_analysis.uncertainties[:, 0].copy()

fixed_run_valid = np.isfinite(fixed_run_distances)
fixed_archive_valid = np.isfinite(fixed_archive_distances)
fixed_common_valid = fixed_run_valid & fixed_archive_valid
fixed_path_difference = fixed_archive_distances[fixed_common_valid] - fixed_run_distances[fixed_common_valid]

print("\nFixed-normal M3C2 results")
print(f"run(): {fixed_run_valid.sum():,}/{len(fixed_run_distances):,} valid")
print(f"archive: {fixed_archive_valid.sum():,}/{len(fixed_archive_distances):,} valid")
print(f"Changed validity: {np.count_nonzero(fixed_run_valid != fixed_archive_valid):,}")
print(
    "Identical distance arrays: "
    f"{np.array_equal(fixed_run_distances, fixed_archive_distances, equal_nan=True)}"
)

if fixed_path_difference.size:
    print(f"Mean difference: {np.mean(fixed_path_difference):.6f} m")
    print(f"MAE difference: {np.mean(np.abs(fixed_path_difference)):.6f} m")
    print(f"RMSE difference: {np.sqrt(np.mean(fixed_path_difference**2)):.6f} m")
    print(f"Maximum absolute difference: {np.max(np.abs(fixed_path_difference)):.6f} m")
else:
    print("No finite distances are shared by both fixed-normal results.")

## results

In [ ]:
# Error: method distance - mesh-reference distance
# Roughness: 0.5 * (spread1 + spread2)
from scipy.spatial import cKDTree
try:
    import pandas as pd
except ImportError:
    pd = None
def _target_column_from_reference(
    reference_analysis,
    target_time,
    reference_time,
):
    expected_delta = target_time - reference_time
    timedeltas = list(reference_analysis.timedeltas)
    matches = [
        idx
        for idx, delta in enumerate(timedeltas)
        if delta == expected_delta
    ]
    if matches:
        return matches[0]
    delta_seconds = np.array([
        abs(
            (delta - expected_delta).total_seconds()
        )
        for delta in timedeltas
    ])
    nearest = int(np.argmin(delta_seconds))
    print(
        f"No exact reference timedelta match for "
        f"{expected_delta}; using nearest column "
        f"{nearest} ({timedeltas[nearest]})."
    )
    return nearest
def _aligned_reference_distance(
    reference_analysis,
    query_corepoints,
    target_time,
    reference_time,
):
    target_col = _target_column_from_reference(
        reference_analysis,
        target_time,
        reference_time,
    )
    reference_corepoints = (
        reference_analysis.corepoints.cloud
    )
    reference_distance_all = (
        reference_analysis.distances[:, target_col]
    )
    same_corepoints = (
        len(reference_corepoints)
        == len(query_corepoints)
        and np.allclose(
            reference_corepoints,
            query_corepoints,
        )
    )
    if same_corepoints:
        return (
            reference_distance_all,
            np.arange(len(query_corepoints)),
            target_col,
        )
    tree = cKDTree(reference_corepoints[:, :3])
    distances_to_ref, reference_idx = tree.query(
        query_corepoints[:, :3],
        k=1,
    )
    print(
        "Corepoints are not identical; using nearest "
        "reference corepoint alignment. "
        f"Median distance="
        f"{np.median(distances_to_ref):.4f} m, "
        f"maximum distance="
        f"{np.max(distances_to_ref):.4f} m"
    )
    return (
        reference_distance_all[reference_idx],
        reference_idx,
        target_col,
    )
def _distance_column(st_analysis, label):
    if (
        st_analysis.distances is None
        or st_analysis.distances.shape[1] == 0
    ):
        raise ValueError(
            f"{label} has no distance column"
        )
    return (
        st_analysis.distances[:, 0]
        .astype(float)
    )
def _distance_vector(distances, label):
    distances = np.asarray(
        distances,
        dtype=float,
    )
    if distances.ndim != 1:
        raise ValueError(
            f"{label} distance array must be 1D, "
            f"got shape {distances.shape}"
        )
    return distances
def _roughness_from_uncertainty_array(
    uncertainty,
    label,
):
    uncertainty = np.asarray(uncertainty)
    names = uncertainty.dtype.names or ()
    if (
        "spread1" not in names
        or "spread2" not in names
    ):
        raise ValueError(
            f"{label} uncertainty does not contain "
            f"spread1/spread2 fields: {names}"
        )
    spread1 = uncertainty[
        "spread1"
    ].astype(float)
    spread2 = uncertainty[
        "spread2"
    ].astype(float)
    return 0.5 * (spread1 + spread2)
def _roughness_from_uncertainty(
    st_analysis,
    label,
):
    if st_analysis.uncertainties is None:
        raise ValueError(
            f"{label} has no uncertainty array"
        )
    return _roughness_from_uncertainty_array(
        st_analysis.uncertainties[:, 0],
        label,
    )
def _finite_percentile(
    values,
    percentile,
    default=1.0,
):
    values = np.asarray(
        values,
        dtype=float,
    )
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return default
    limit = float(
        np.nanpercentile(
            finite,
            percentile,
        )
    )
    if not np.isfinite(limit) or limit == 0:
        return default
    return limit
def _format_map_axes(ax):
    ax.set_xlabel("X [m]")
    ax.set_ylabel("Y [m]")
    ax.set_aspect(
        "equal",
        adjustable="box",
    )
reference_distance, reference_indices, reference_target_col = (
    _aligned_reference_distance(
        ref_analysis,
        corepoints,
        target_timestamp,
        reference_timestamp,
    )
)
print(
    f"Using reference distance column "
    f"{reference_target_col} for target "
    f"{target_timestamp}"
)
comparison_results = {
    # Original archive result
    "Direct M3C2 archive": {
        "distance": _distance_column(
            direct_m3c2_analysis,
            "Direct M3C2 archive",
        ),
        "roughness": _roughness_from_uncertainty(
            direct_m3c2_analysis,
            "Direct M3C2 archive",
        ),
    },
    # Original run() result
    "Direct M3C2 run()": {
        "distance": _distance_vector(
            m3c2_distances,
            "Direct M3C2 run()",
        ),
        "roughness": (
            _roughness_from_uncertainty_array(
                m3c2_uncertainties,
                "Direct M3C2 run()",
            )
        ),
    },
    # Additional fixed-normal run() result
    "Fixed-normal M3C2 run()": {
        "distance": _distance_vector(
            fixed_run_distances,
            "Fixed-normal M3C2 run()",
        ),
        "roughness": (
            _roughness_from_uncertainty_array(
                fixed_run_uncertainties,
                "Fixed-normal M3C2 run()",
            )
        ),
    },
    # Additional fixed-normal archive result
    "Fixed-normal M3C2 archive": {
        "distance": _distance_vector(
            fixed_archive_distances,
            "Fixed-normal M3C2 archive",
        ),
        "roughness": (
            _roughness_from_uncertainty_array(
                fixed_archive_uncertainties,
                "Fixed-normal M3C2 archive",
            )
        ),
    },
    "Best scale unweighted": {
        "distance": _distance_column(
            no_weight_analysis,
            "Best scale unweighted",
        ),
        "roughness": _roughness_from_uncertainty(
            no_weight_analysis,
            "Best scale unweighted",
        ),
    },
    "Best scale temporal-only": {
        "distance": _distance_column(
            temporal_only_analysis,
            "Best scale temporal-only",
        ),
        "roughness": _roughness_from_uncertainty(
            temporal_only_analysis,
            "Best scale temporal-only",
        ),
    },
    "Multiscale weighted": {
        "distance": _distance_column(
            single_target_multiscale_analysis,
            "Multiscale weighted",
        ),
        "roughness": _roughness_from_uncertainty(
            single_target_multiscale_analysis,
            "Multiscale weighted",
        ),
    },
    "Best scale weighted": {
        "distance": _distance_column(
            best_scale_analysis,
            "Best scale weighted",
        ),
        "roughness": _roughness_from_uncertainty(
            best_scale_analysis,
            "Best scale weighted",
        ),
    },
}
comparison_summary_rows = []
for label, result in comparison_results.items():
    result["error"] = (
        result["distance"]
        - reference_distance
    )
    valid_error = np.isfinite(
        result["error"]
    )
    valid_roughness = np.isfinite(
        result["roughness"]
    )
    error = result["error"][valid_error]
    summary_row = {
        "method": label,
        "valid_error_corepoints": int(
            valid_error.sum()
        ),
        "total_corepoints": int(
            len(valid_error)
        ),
        "mean_error_m": (
            float(np.mean(error))
            if error.size
            else np.nan
        ),
        "mae_m": (
            float(np.mean(np.abs(error)))
            if error.size
            else np.nan
        ),
        "rmse_m": (
            float(
                np.sqrt(
                    np.mean(error**2)
                )
            )
            if error.size
            else np.nan
        ),
        "std_error_m": (
            float(np.std(error))
            if error.size
            else np.nan
        ),
        "mean_roughness_m": float(
            np.nanmean(result["roughness"])
        ),
        "valid_roughness_corepoints": int(
            valid_roughness.sum()
        ),
    }
    comparison_summary_rows.append(
        summary_row
    )
    print(
        f"{label}: "
        f"valid error="
        f"{summary_row['valid_error_corepoints']:,}/"
        f"{summary_row['total_corepoints']:,}, "
        f"mean error="
        f"{summary_row['mean_error_m']:.4f} m, "
        f"MAE="
        f"{summary_row['mae_m']:.4f} m, "
        f"RMSE="
        f"{summary_row['rmse_m']:.4f} m, "
        f"STD="
        f"{summary_row['std_error_m']:.4f} m, "
        f"mean roughness="
        f"{summary_row['mean_roughness_m']:.4f} m "
        f"({summary_row['valid_roughness_corepoints']:,} valid)"
    )
common_valid_error = np.logical_and.reduce([
    np.isfinite(result["error"])
    for result in comparison_results.values()
])
print(
    "\nCommon valid-error corepoints: "
    f"{common_valid_error.sum():,}/"
    f"{len(common_valid_error):,} "
    "valid for every method and the reference"
)
common_summary_rows = []
for label, result in comparison_results.items():
    common_error = result[
        "error"
    ][common_valid_error]
    common_summary = {
        "method": label,
        "common_valid_corepoints": int(
            common_valid_error.sum()
        ),
        "common_mean_error_m": float(
            np.mean(common_error)
        ),
        "common_mae_m": float(
            np.mean(np.abs(common_error))
        ),
        "common_rmse_m": float(
            np.sqrt(
                np.mean(common_error**2)
            )
        ),
        "common_std_error_m": float(
            np.std(common_error)
        ),
    }
    common_summary_rows.append(
        common_summary
    )
    print(
        f"{label} (common valid): "
        f"mean error="
        f"{common_summary['common_mean_error_m']:.4f} m, "
        f"MAE="
        f"{common_summary['common_mae_m']:.4f} m, "
        f"RMSE="
        f"{common_summary['common_rmse_m']:.4f} m, "
        f"STD="
        f"{common_summary['common_std_error_m']:.4f} m"
    )
if pd is not None:
    comparison_summary_df = pd.DataFrame(
        comparison_summary_rows
    )
    common_summary_df = pd.DataFrame(
        common_summary_rows
    )
    display(comparison_summary_df)
    display(common_summary_df)
xy = corepoints[:, :2]
error_values = np.concatenate([
    result["error"]
    for result in comparison_results.values()
])
roughness_values = np.concatenate([
    result["roughness"]
    for result in comparison_results.values()
])
error_limit = _finite_percentile(
    np.abs(error_values),
    98,
    default=0.1,
)
roughness_limit = _finite_percentile(
    roughness_values,
    98,
    default=0.1,
)
number_of_methods = len(
    comparison_results
)
fig, axs = plt.subplots(
    2,
    4,
    figsize=(22, 10),
    constrained_layout=True,
)
for ax, (label, result) in zip(
    np.ravel(axs),
    comparison_results.items(),
):
    scatter = ax.scatter(
        xy[:, 0],
        xy[:, 1],
        c=result["error"],
        s=3,
        cmap="seismic_r",
        vmin=-error_limit,
        vmax=error_limit,
    )
    ax.set_title(label, fontsize=12)
    _format_map_axes(ax)
    fig.colorbar(
        scatter,
        ax=ax,
        label="Error [m]",
        fraction=0.046,
        pad=0.04,
    )
plt.show()
fig, axs = plt.subplots(
    2,
    4,
    figsize=(22, 10),
    constrained_layout=True,
)
for ax, (label, result) in zip(
    np.ravel(axs),
    comparison_results.items(),
):
    scatter = ax.scatter(
        xy[:, 0],
        xy[:, 1],
        c=result["roughness"],
        s=3,
        cmap="viridis",
        vmin=0,
        vmax=roughness_limit,
    )
    ax.set_title(
        f"{label}\n"
        "roughness = mean(spread1, spread2)",
        fontsize=12,
    )
    _format_map_axes(ax)
    fig.colorbar(
        scatter,
        ax=ax,
        label="Spread roughness [m]",
        fraction=0.046,
        pad=0.04,
    )
plt.show()

## run time series

In [ ]:
# Run the full time series using the globally selected best single scale.
timeseries_best_scale_tam = TAM3C2(
    epochs_timeseries=epochs_all,
    max_window_ratio=float(best_wr),
    normal_radii=float(best_sr),
    required_points=required_points,
    weighting=weighting,
    sigma_ratio=sigma_ratio,
    space_time_ratio=space_time_ratio,
    include_center_epoch=include_center_epoch,
    keep_neighborhoods=False,
    corepoints=corepoints,
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

timeseries_best_scale_analysis = py4dgeo.SpatiotemporalAnalysis(timeseries_output_path,force=True,)
timeseries_best_scale_analysis.reference_epoch = reference_epoch
timeseries_best_scale_analysis.corepoints = corepoints
timeseries_best_scale_analysis.m3c2 = timeseries_best_scale_tam

timeseries_target_epochs = [
    epoch
    for epoch in epochs_all
    if epoch.timestamp != reference_epoch.timestamp
]
timeseries_best_scale_analysis.add_epochs(*timeseries_target_epochs)

# Standard M3C2 full time series using the same spatial parameters.
timeseries_standard_m3c2 = py4dgeo.M3C2(
    epochs=(reference_epoch, target_epoch),
    corepoints=corepoints,
    normal_radii=[float(best_sr)],
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

timeseries_standard_m3c2_analysis = py4dgeo.SpatiotemporalAnalysis(
    timeseries_standard_m3c2_output_path,
    force=True,
)
timeseries_standard_m3c2_analysis.reference_epoch = reference_epoch
timeseries_standard_m3c2_analysis.corepoints = corepoints
timeseries_standard_m3c2_analysis.m3c2 = timeseries_standard_m3c2

timeseries_standard_m3c2_analysis.add_epochs(*timeseries_target_epochs)

print(f"Best single scale: normal_radius={best_sr}, max_window_ratio={best_wr}")
print(f"Shared M3C2 parameters: cyl_radius={cyl_radius}, max_distance={max_distance}, registration_error={registration_error}")
print(f"Saved TAM3C2 full time series: {timeseries_output_path}")
print(f"Saved standard M3C2 full time series: {timeseries_standard_m3c2_output_path}")
print(f"Added epochs: {len(timeseries_target_epochs)}")
print(f"TAM3C2 distances shape: {timeseries_best_scale_analysis.distances.shape}")
print(f"Standard M3C2 distances shape: {timeseries_standard_m3c2_analysis.distances.shape}")

In [ ]:
# Run the 4D-OBC algorithm
algo = RegionGrowingAlgorithm(
    neighborhood_radius=obc_neighborhood_radius,
    min_segments=obc_min_segments,
    minperiod=obc_minperiod,
    height_threshold=obc_height_threshold,
    thresholds=obc_thresholds,
    seed_subsampling=1,
)

timeseries_standard_m3c2_analysis.invalidate_results(seeds=True, objects=True)
m3c2_objects = algo.run(timeseries_standard_m3c2_analysis)
print(f"Extracted {len(m3c2_objects)} 4D-OBCs from {len(timeseries_standard_m3c2_analysis.seeds)} seeds")


timeseries_best_scale_analysis.invalidate_results(seeds=True, objects=True)
tam3c2_objects = algo.run(timeseries_best_scale_analysis)
print(f"Extracted {len(tam3c2_objects)} 4D-OBCs from {len(timeseries_best_scale_analysis.seeds)} seeds")

In [ ]:
print(f"Extracted {len(m3c2_objects)} 4D-OBCs from {len(timeseries_standard_m3c2_analysis.seeds)} seeds")


In [ ]:
print(f"Extracted {len(tam3c2_objects)} 4D-OBCs from {len(timeseries_best_scale_analysis.seeds)} seeds")